In [2]:
import yfinance as yf
import pandas as pd
import ta
import talib
import numpy as np

# ── tickers ───────────────────────────────────────────────
tickers = ["RELIANCE.NS", "TCS.NS", "INFY.NS",
           "HDFCBANK.NS", "SBIN.NS"]

START = "2018-01-01"
END   = "2026-06-12"

# ── download nifty 50 (market context) ───────────────────
print("Downloading Nifty 50...")
nifty_raw = yf.download("^NSEI", start=START, end=END, auto_adjust=True)

if isinstance(nifty_raw.columns, pd.MultiIndex):
    nifty_raw.columns = nifty_raw.columns.get_level_values(0)

nifty_raw["nifty_return_1d"] = nifty_raw["Close"].pct_change(1)
nifty_raw["nifty_return_5d"] = nifty_raw["Close"].pct_change(5)
nifty_raw["nifty_rsi"]       = ta.momentum.RSIIndicator(
    nifty_raw["Close"], window=14).rsi()
nifty_raw["nifty_above_ema"] = (
    nifty_raw["Close"] > ta.trend.EMAIndicator(
        nifty_raw["Close"], window=50).ema_indicator()
).astype(int)

nifty_features = nifty_raw[[
    "nifty_return_1d", "nifty_return_5d",
    "nifty_rsi", "nifty_above_ema"
]].copy()

print(f"Nifty features shape: {nifty_features.shape}")

# ── download stock data ───────────────────────────────────
all_data = {}

for ticker in tickers:
    print(f"Downloading {ticker}...")
    df = yf.download(ticker, start=START, end=END, auto_adjust=True)

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df["ticker"] = ticker
    all_data[ticker] = df
    print(f"  {ticker}: {len(df)} rows")

print("\nAll stocks downloaded")

[*********************100%***********************]  1 of 1 completed


Nifty features shape: (2079, 4)


[*********************100%***********************]  1 of 1 completed


  RELIANCE.NS: 2086 rows


[*********************100%***********************]  1 of 1 completed


...

  INFY.NS: 2086 rows


[*********************100%***********************]  1 of 1 completed


  HDFCBANK.NS: 2086 rows


[*********************100%***********************]  1 of 1 completed

  SBIN.NS: 2086 rows

All stocks downloaded


In [17]:
def add_candlestick_patterns(df):
    o = df["Open"].values
    h = df["High"].values
    l = df["Low"].values
    c = df["Close"].values

    df["cdl_doji"]            = talib.CDLDOJI(o, h, l, c)
    df["cdl_hammer"]          = talib.CDLHAMMER(o, h, l, c)
    df["cdl_engulfing"]       = talib.CDLENGULFING(o, h, l, c)
    df["cdl_shooting_star"]   = talib.CDLSHOOTINGSTAR(o, h, l, c)
    df["cdl_morning_star"]    = talib.CDLMORNINGSTAR(o, h, l, c)
    df["cdl_evening_star"]    = talib.CDLEVENINGSTAR(o, h, l, c)
    df["cdl_marubozu"]        = talib.CDLMARUBOZU(o, h, l, c)
    df["cdl_3white_soldiers"] = talib.CDL3WHITESOLDIERS(o, h, l, c)

    pattern_cols = [col for col in df.columns if col.startswith("cdl_")]
    df[pattern_cols] = df[pattern_cols] / 100

    df["pattern_strength"] = df[pattern_cols].sum(axis=1)
    df["pattern_count"]    = (df[pattern_cols] != 0).sum(axis=1)
    return df


def add_features(df, nifty_features):

    # ==========================================
    # MOMENTUM
    # ==========================================

    df["rsi"] = ta.momentum.RSIIndicator(
        df["Close"],
        window=14
    ).rsi()

    # ==========================================
    # MACD
    # ==========================================

    macd = ta.trend.MACD(
        df["Close"],
        window_fast=12,
        window_slow=26,
        window_sign=9
    )

    df["macd"] = macd.macd()
    df["macd_signal"] = macd.macd_signal()
    df["macd_hist"] = macd.macd_diff()

    # ==========================================
    # EMA
    # ==========================================

    df["ema_20"] = ta.trend.EMAIndicator(
        df["Close"],
        window=20
    ).ema_indicator()

    df["ema_50"] = ta.trend.EMAIndicator(
        df["Close"],
        window=50
    ).ema_indicator()

    df["ema_cross"] = (
        df["ema_20"] > df["ema_50"]
    ).astype(int)

    # ==========================================
    # BOLLINGER BANDS
    # ==========================================

    bb = ta.volatility.BollingerBands(
        df["Close"],
        window=20,
        window_dev=2
    )

    df["bb_upper"] = bb.bollinger_hband()
    df["bb_lower"] = bb.bollinger_lband()

    df["bb_width"] = (
        df["bb_upper"] - df["bb_lower"]
    ) / df["Close"]

    # ==========================================
    # ATR
    # ==========================================

    df["atr"] = ta.volatility.AverageTrueRange(
        df["High"],
        df["Low"],
        df["Close"],
        window=14
    ).average_true_range()

    # ==========================================
    # VOLUME
    # ==========================================

    df["volume_ma"] = (
        df["Volume"]
        .rolling(20)
        .mean()
    )

    df["volume_ratio"] = (
        df["Volume"]
        / df["volume_ma"]
    )

    # ==========================================
    # RETURNS
    # ==========================================

    df["return_1d"] = df["Close"].pct_change(1)
    df["return_5d"] = df["Close"].pct_change(5)
    df["return_10d"] = df["Close"].pct_change(10)

    df["lag_1"] = df["return_1d"].shift(1)
    df["lag_2"] = df["return_1d"].shift(2)
    df["lag_3"] = df["return_1d"].shift(3)

    # ==========================================
    # CANDLESTICK PATTERNS
    # ==========================================

    df = add_candlestick_patterns(df)

    # ==========================================
    # NIFTY FEATURES (SAFE VERSION)
    # ==========================================

    df["nifty_return_1d"] = nifty_features["nifty_return_1d"]
    df["nifty_return_5d"] = nifty_features["nifty_return_5d"]
    df["nifty_rsi"] = nifty_features["nifty_rsi"]
    df["nifty_above_ema"] = nifty_features["nifty_above_ema"]

    df["rel_strength_1d"] = (
        df["return_1d"]
        - df["nifty_return_1d"]
    )

    df["rel_strength_5d"] = (
        df["return_5d"]
        - df["nifty_return_5d"]
    )

    df["rsi_vs_nifty"] = (
        df["rsi"]
        - df["nifty_rsi"]
    )

    # ==========================================
    # TARGET (5-DAY DIRECTION)
    # ==========================================

    df["future_close_5d"] = (
        df["Close"]
        .shift(-5)
    )

    df["target"] = (
        df["future_close_5d"]
        > df["Close"]
    ).astype(int)

    # ==========================================
    # CLEANUP
    # ==========================================

    df.drop(
        columns=["future_close_5d"],
        inplace=True,
        errors="ignore"
    )

    df.dropna(inplace=True)

    return df

In [18]:
print(nifty_features.columns)

df = df.merge(
    nifty_features,
    left_index=True,
    right_index=True,
    how="left"
)

print(df.filter(regex="nifty").columns)

Index(['nifty_return_1d', 'nifty_return_5d', 'nifty_rsi', 'nifty_above_ema'], dtype='str')
Index(['nifty_return_1d_x', 'nifty_return_5d_x', 'nifty_rsi_x',
       'nifty_above_ema_x', 'nifty_return_1d_y', 'nifty_return_5d_y',
       'nifty_rsi_y', 'nifty_above_ema_y'],
      dtype='str')


In [19]:
nifty_features.columns.name = None

In [20]:
for ticker in tickers:
    all_data[ticker] = add_features(all_data[ticker], nifty_features)
    print(f"{ticker}: {len(all_data[ticker])} rows, "
          f"{len(all_data[ticker].columns)} columns")

final_df = pd.concat(all_data.values())
final_df.reset_index(inplace=True)
final_df.rename(columns={"index": "Date"}, inplace=True)

print(f"\nTotal rows    : {len(final_df)}")
print(f"Total columns : {len(final_df.columns)}")

# verify no NaNs
nan_check = final_df.isnull().sum()
print(f"NaN columns   : {nan_check[nan_check > 0].to_dict()}")

final_df.to_csv("all_stocks_features.csv", index=False)
print("Saved data/all_stocks_features.csv")

RELIANCE.NS: 1982 rows, 43 columns
TCS.NS: 1982 rows, 43 columns
INFY.NS: 1982 rows, 43 columns
HDFCBANK.NS: 1982 rows, 43 columns
SBIN.NS: 1982 rows, 43 columns

Total rows    : 9910
Total columns : 44
NaN columns   : {}
Saved data/all_stocks_features.csv


In [21]:
print(final_df.shape)

print(final_df["target"].value_counts())

print(
    final_df["target"]
    .value_counts(normalize=True)
)

(9910, 44)
target
1    5351
0    4559
Name: count, dtype: int64
target
1    0.53996
0    0.46004
Name: proportion, dtype: float64


In [23]:
df_check = pd.read_csv("all_stocks_features.csv")
df_check = pd.get_dummies(df_check, columns=["ticker"], dtype=int)

DROP_COLS = ["Date", "Open", "High", "Low", "Close", "Volume", "ticker", "target"]
feat_cols = [c for c in df_check.columns if c not in DROP_COLS]

corr = df_check[feat_cols + ["target"]].corr()["target"].drop("target")

print("Top 10 correlations with 5-day target:")
print(corr.abs().sort_values(ascending=False).head(10))

print("\nNifty feature correlations:")
nifty_cols = ["nifty_return_1d", "nifty_return_5d",
              "nifty_rsi", "nifty_above_ema",
              "rel_strength_1d", "rel_strength_5d", "rsi_vs_nifty"]
for col in nifty_cols:
    if col in corr.index:
        print(f"  {col:<22}: {corr[col]:.4f}")

Top 10 correlations with 5-day target:
bb_upper              0.043372
ema_20                0.042649
bb_lower              0.042375
ema_50                0.041981
atr                   0.040593
rsi_vs_nifty          0.027758
ticker_HDFCBANK.NS    0.020854
pattern_count         0.019718
ticker_SBIN.NS        0.019639
ticker_TCS.NS         0.018323
Name: target, dtype: float64

Nifty feature correlations:
  nifty_return_1d       : 0.0075
  nifty_return_5d       : 0.0114
  nifty_rsi             : 0.0181
  nifty_above_ema       : 0.0137
  rel_strength_1d       : -0.0059
  rel_strength_5d       : -0.0040
  rsi_vs_nifty          : -0.0278


In [25]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [28]:
import pandas as pd

df = pd.read_csv("all_stocks_features.csv")

df["Date"] = pd.to_datetime(df["Date"])

print(df.shape)
print("target" in df.columns)

(9910, 44)
True


In [29]:
from src.data_loader import load_data

(
    X_train,
    X_test,
    y_train,
    y_test,
    FEATURE_COLS
) = load_data()

print(df.shape)

print(df["target"].value_counts(normalize=True))

print(df["Date"].min())
print(df["Date"].max())

Data range : 2018-05-25 → 2026-06-11
Total rows : 9910
Total Features : 41
Features       : ['rsi', 'macd', 'macd_signal', 'macd_hist', 'ema_20', 'ema_50', 'ema_cross', 'bb_upper', 'bb_lower', 'bb_width', 'atr', 'volume_ma', 'volume_ratio', 'return_1d', 'return_5d', 'return_10d', 'lag_1', 'lag_2', 'lag_3', 'cdl_doji', 'cdl_hammer', 'cdl_engulfing', 'cdl_shooting_star', 'cdl_morning_star', 'cdl_evening_star', 'cdl_marubozu', 'cdl_3white_soldiers', 'pattern_strength', 'pattern_count', 'nifty_return_1d', 'nifty_return_5d', 'nifty_rsi', 'nifty_above_ema', 'rel_strength_1d', 'rel_strength_5d', 'rsi_vs_nifty', 'ticker_HDFCBANK.NS', 'ticker_INFY.NS', 'ticker_RELIANCE.NS', 'ticker_SBIN.NS', 'ticker_TCS.NS']

Train : 7430 rows | 2018-05-25 → 2024-06-07
Test  : 2480 rows  | 2024-06-10 → 2026-06-11

Train UP% : 55.1%
Test  UP% : 50.6%

Scaler and feature_cols saved
(9910, 44)
target
1    0.53996
0    0.46004
Name: proportion, dtype: float64
2018-05-25 00:00:00
2026-06-11 00:00:00
